# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object follows the Dataset schema; access fields as attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Let's review available record sets, fields, and their `@id`s as defined in the Croissant package.

We'll list all record sets and then all fields within each record set, referencing them by their `@id` values. This allows referencing specific data elements in a schema-compliant way.

In [ ]:
# Retrieve all record sets in the dataset by their @id

record_sets = [rs for rs in dataset.record_sets]

if not record_sets:
    print("No record sets found in dataset; attempting to list columns and top-level fields if present.")
    
    # List all possible fields in the top-level dataset (fallback in case record_sets are empty)
    if hasattr(meta, 'fields'):
        print("Top-level fields (by @id):")
        for field in meta.fields:
            print(f" - {field['@id'] if '@id' in field else field}")
else:
    print("Record sets and associated fields (by @id):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):  # Single field
                print(f"  - Field: {fields['@id']}")
            elif isinstance(fields, list):
                for field in fields:
                    if isinstance(field, dict) and '@id' in field:
                        print(f"  - Field: {field['@id']}")
                    else:
                        print(f"  - Field: {field}")
            else:
                print(f"  - Field: {fields}")
        else:
            print("  - No fields listed.")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s found above.

If record sets are not defined, we'll attempt to load all records (as Free Table) using the default method.

In [ ]:
# List all accessible record set @ids
record_set_ids = []
for rs in dataset.record_sets:
    if '@id' in rs:
        record_set_ids.append(rs['@id'])

# If the schema doesn't define record sets, use 'default' to access all records.
if not record_set_ids:
    print("No explicit record sets; attempting to fetch records using the default loader.")
    default_record_set_id = None
    try:
        # Try to get a list of records, and guess columns
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print("Failed to load records: ", e)
else:
    # Dictionary of DataFrames for each record set
    dataframes = {}
    for rsid in record_set_ids:
        print(f"\nLoading records for record set @id: {rsid}")
        try:
            records = list(dataset.records(record_set=rsid))
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records. Columns:")
            print(df.columns.tolist())
            print(df.head())
        except Exception as e:
            print(f"Failed to load record set {rsid}: {e}")
    # Example: print columns and head of first record set
    if dataframes:
        first_rsid = record_set_ids[0]
        print(f"\nExample columns in record set {first_rsid}:\n{dataframes[first_rsid].columns.tolist()}")
        display(dataframes[first_rsid].head())


## 4. Exploratory Data Analysis (EDA)
Apply some prototypical data processing steps, such as filtering records based on a numeric field, normalizing, and grouping by a categorical field. All fields are referenced by their `@id`.

**Choose a numeric field (for example, patient age or diagnosis interval) and a group field (such as sex or cancer subsite) for demonstration.**

In [ ]:
# If we loaded with record_sets:
if 'dataframes' in locals() and dataframes:
    # Pick the first record set for EDA
    rsid = list(dataframes.keys())[0]
    df = dataframes[rsid]
else:
    # Fallback on default-loaded DataFrame (df)
    rsid = None

# Try to infer numeric and group fields by @id (column name)
print("Columns available:", df.columns.tolist())

# Adjust these IDs to match columns in your data (use print above)
# Example guesses - if using real field @id, replace accordingly
# Let's suppose:
#  - Age @id: 'schema:age'
#  - Sex @id: 'schema:sex'
#  - Diagnosis Interval @id: 'diagnosisInterval'

possible_numeric_fields = [col for col in df.columns if 'ge' in col.lower() or 'interval' in col.lower() or 'year' in col.lower()]
possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'site' in col.lower() or 'msi' in col.lower()]

# Choose first found, or fallback to manual guess
numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
group_field_id = possible_group_fields[0] if possible_group_fields else df.columns[1]
print(f"Numeric field candidate (@id): {numeric_field_id}")
print(f"Group field candidate (@id): {group_field_id}")

# Filter: e.g., age > 50 or interval > 0
threshold = 50 if 'age' in numeric_field_id.lower() else 1
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (first rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id if available
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll provide a histogram for the numeric field and a boxplot grouped by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group
plt.figure(figsize=(8,5))
sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
plt.title(f"{numeric_field_id} by {group_field_id}")
plt.xlabel(group_field_id)
plt.ylabel(numeric_field_id)
plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset via its Croissant schema with `mlcroissant`, listed available record sets and fields using their `@id`s, extracted records into a DataFrame, and performed filtering, normalization, grouping, and visualization. This provides a reproducible template for exploring FAIR datasets and analyzing clinical/biomarker data using standard Python data science tools.